# 2. Bring your own model and data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Explainable-AI-Berlin/pytorch_explain_and_adapt_library/blob/master/notebooks/02_bring_your_own_model_and_data.ipynb)

PEAL never needs to train your classifier. It needs two things from you:

1. **images** in a two-class layout PEAL can read, and
2. **a model** it can call: a pickled torch module (`.cpl`) or an **ONNX** file.

This notebook shows both contracts with a working example and checks that model and data fit
together before any expensive explanation runs. Nothing here needs a GPU for long: the example model
is a pretrained ImageNet ResNet-18 cut down to two classes, and the images are the freight cars and
passenger cars of notebook 00. Replace either with your own; notebooks 01 and 03 then explain the
model (with SCE and with DiDAE).

**The one thing that goes wrong most often:** PEAL feeds the model exactly the tensor the data config
produces. The image size and normalization your model expects have to be written into the
`DataConfig` (`input_size`, `normalization`), or baked into the exported graph.


## 0. Setup

The same setup cell as in every PEAL notebook (installs PEAL on Colab, finds the configs, sets
`PEAL_DATA` / `PEAL_RUNS`).


In [1]:
# Setup: the same cell in every PEAL notebook. Works on Google Colab and locally.
# Colab: Runtime -> Change runtime type -> a GPU. The first run installs PEAL and
# restarts the runtime once (PEAL needs numpy < 2); then run this cell again.
import importlib.util, logging, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PEAL_VERSION = "0.1.0"
PEAL_REPO = "https://github.com/Explainable-AI-Berlin/pytorch_explain_and_adapt_library.git"

if importlib.util.find_spec("peal") is None:
    # --ignore-requires-python: PEAL 0.1.0 declares Python < 3.12, Colab runs 3.12
    # (it works there). OpenAI's `clip` package is not on PyPI; the RAE generator needs it.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                    f"peal-xai[rae,onnx]=={PEAL_VERSION}", "git+https://github.com/openai/CLIP.git"],
                   check=True)
    if IN_COLAB:
        print("PEAL installed. Restarting the runtime once; run this cell again afterwards.")
        os.kill(os.getpid(), 9)


def find_configs():
    """PEAL's config files live in its repository, not in the pip package.
    Use $PEAL_BASE, else the repository this notebook sits in, else a sparse clone
    (configs/ and the dictionary vocabulary only)."""
    if os.environ.get("PEAL_BASE"):
        return Path(os.environ["PEAL_BASE"])
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "configs" / "web_demo").is_dir():
            return parent
    target = Path("/content/peal" if IN_COLAB else Path.home() / "peal_configs")
    if not (target / "configs").is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                        PEAL_REPO, str(target)], check=True)
        subprocess.run(["git", "-C", str(target), "sparse-checkout", "set", "configs",
                        "peal/dependencies/MSAE/vocab"], check=True)
    return target


os.environ["PEAL_BASE"] = str(find_configs())            # <PEAL_BASE> in config files
CONFIGS = Path(os.environ["PEAL_BASE"]) / "configs"
ROOT = Path("/content/peal_data") if IN_COLAB else Path.home() / "peal_data"
os.environ.setdefault("PEAL_DATA", str(ROOT / "datasets"))  # $PEAL_DATA: datasets
os.environ.setdefault("PEAL_RUNS", str(ROOT / "runs"))      # $PEAL_RUNS: models and runs
os.environ.setdefault("PEAL_RAE_WEIGHTS", "hf://sidney1505/peal-rae-clip-imagenet@v0.1.0")
DATA, RUNS = Path(os.environ["PEAL_DATA"]), Path(os.environ["PEAL_RUNS"])

import warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")      # TensorFlow (pulled in by a dependency) is chatty
warnings.filterwarnings("ignore", message="To copy construct from a tensor")       # PEAL 0.1.0 dataset noise
warnings.filterwarnings("ignore", message='Field "model_', category=UserWarning)   # pydantic namespace notes
logging.getLogger("matplotlib.image").setLevel(logging.ERROR)                      # "Clipping input data" notes

import peal, shutil, torch
# PEAL 0.1.0's wheel misses the concept vocabulary of the MSAE dictionary; copy it in from the repo
from peal.sparse_dictionaries.msae_decomposition import MSAE_DIR
_vocab = Path(MSAE_DIR) / "vocab" / "clip_disect_20k.txt"
if not _vocab.exists():
    _vocab.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(Path(os.environ["PEAL_BASE"]) / "peal/dependencies/MSAE/vocab/clip_disect_20k.txt", _vocab)
from peal.log import configure
configure()   # PEAL logs progress at INFO; drop only its "Loading config ..." dumps and file lists
_quiet = lambda r: not str(r.getMessage()).startswith(
    ("Loading config", "Config model", "No config model", "Saved collage", "[DiDAE Sweep] decoded "))
for _h in logging.getLogger("peal").handlers:
    _h.addFilter(_quiet)
print(f"PEAL {peal.__version__} | configs: {CONFIGS}")
print(f"PEAL_DATA: {DATA} | PEAL_RUNS: {RUNS}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

# ---- notebook-specific -------------------------------------------------------------------
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

WORK = RUNS / "bring_your_own"
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


PEAL 0.1.0 | configs: /home/space/datasets/peal/peal_runs/notebook_sim/peal_base/configs
PEAL_DATA: /home/space/datasets/peal/peal_runs/notebook_sim/data500 | PEAL_RUNS: /home/space/datasets/peal/peal_runs/notebook_sim/runs_final4
GPU: NVIDIA H100 PCIe


## 1. Your images: one folder per class

The most natural way to hand over images is one folder per class, exactly what the PEAL web demo
accepts as a zip:

```
my_images/
    freight car/     *.jpg
    passenger car/   *.jpg
```

`peal.web.ingest` turns that into the layout PEAL's generic `Image2MixedDataset` reads
(`imgs/0/...`, `imgs/1/...` and a `data.csv` with the header `ImgPath,Class`). `discover_classes`
finds the class folders (also one or two levels down, as zips often have), `build_pair_dataset` picks
two of them, label 0 and label 1, and symlinks the images. `ingest_zip` does the same for a zip file,
with the safety checks the web demo needs for uploads from strangers.

### Option A (default): the freight and passenger cars from notebook 00

If you ran notebook 00 they are already on disk; otherwise this cell fetches them the same way
(the two ImageNet classes from Hugging Face; accept the terms of `ILSVRC/imagenet-1k` and log in).
It then writes them as one folder per class, so the rest looks exactly like your own data.


In [2]:
TRAINS = DATA / "freight_car_vs_passenger_car"
if not (TRAINS / "data.csv").exists():
    from datasets import load_dataset   # preinstalled on Colab; else pip install datasets
    counts, rows = {565: 0, 705: 0}, []
    for ex in load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True):
        k = ex["label"]
        if k in counts and counts[k] < 50:
            label = {565: 0, 705: 1}[k]
            (TRAINS / "imgs" / str(label)).mkdir(parents=True, exist_ok=True)
            rel = f"{label}/{k}_{counts[k]:03d}.jpg"
            ex["image"].convert("RGB").save(TRAINS / "imgs" / rel, quality=95)
            rows.append((rel, label)); counts[k] += 1
            if min(counts.values()) >= 50:
                break
    pd.DataFrame(rows, columns=["ImgPath", "Class"]).to_csv(TRAINS / "data.csv", index=False)

# one folder per class, named after the class - the format you would bring yourself
MY_IMAGES = WORK / "my_images"
for label, name in enumerate(["freight car", "passenger car"]):
    (MY_IMAGES / name).mkdir(parents=True, exist_ok=True)
    for src in sorted((TRAINS / "imgs" / str(label)).iterdir()):
        dst = MY_IMAGES / name / src.name
        if not dst.exists():
            dst.symlink_to(src.resolve())
print({p.name: len(list(p.iterdir())) for p in sorted(MY_IMAGES.iterdir())})


{'freight car': 500, 'passenger car': 500}


### Option B: your own folder (or zip)

Point `MY_IMAGES` at your folder, or use `ingest_zip(zip_path, work_dir)` for a zip; both return the
class folders found.


In [3]:
USE_OWN_IMAGES = False
if USE_OWN_IMAGES:
    MY_IMAGES = Path("/path/to/my_images")          # one sub-folder per class
    # from peal.web.ingest import ingest_zip
    # class_root, classes = ingest_zip("/path/to/my_images.zip", str(WORK / "zip"))


In [4]:
from peal.web.ingest import discover_classes, build_pair_dataset

class_root, classes = discover_classes(str(MY_IMAGES))
print("classes found:", {name: len(files) for name, files in classes.items()})
CLASS_A, CLASS_B = "freight car", "passenger car"     # label 0 and label 1
DATASET = DATA / "my_two_classes"
counts = build_pair_dataset(class_root, classes, CLASS_A, CLASS_B, str(DATASET))
print(counts, "->", DATASET)
print(open(DATASET / "data.csv").read().splitlines()[:3])


classes found: {'freight car': 500, 'passenger car': 500}


{'freight car': 500, 'passenger car': 500} -> /home/space/datasets/peal/peal_runs/notebook_sim/data500/my_two_classes
['ImgPath,Class', '0/000000.jpeg,0', '0/000001.jpeg,0']


## 2. Your model as ONNX

Any framework that exports ONNX works (PyTorch, TensorFlow via `tf2onnx`, scikit-learn via
`skl2onnx`, ...). Three rules:

* **two outputs**, the logits of label 0 and label 1, in that order;
* a **dynamic batch axis** - PEAL calls the graph in batches;
* the input is **exactly what the data config produces**: `float32 [N, 3, H, W]`, RGB, after the
  normalization you put into the config.

### Option A (default): a pretrained ImageNet ResNet-18 as a freight-car classifier

ImageNet already has both classes (565 freight car, 705 passenger car), so we keep only those two
logits of a pretrained ResNet-18. This model expects ImageNet-normalized images; we leave the
normalization *outside* the graph and tell PEAL about it in the data config (section 3).


In [5]:
import torchvision

class TwoClassesOf(torch.nn.Module):
    """Keep two logits of an ImageNet classifier: [class_a, class_b]."""
    def __init__(self, model, class_a, class_b):
        super().__init__()
        self.model, self.keep = model, [class_a, class_b]
    def forward(self, x):
        return self.model(x)[:, self.keep]

import warnings

model = TwoClassesOf(torchvision.models.resnet18(weights="IMAGENET1K_V1"), 565, 705).eval()
onnx_path = WORK / "freight_car_resnet18.onnx"
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)    # we choose the classic exporter on purpose
    torch.onnx.export(
        model, torch.randn(1, 3, 224, 224), str(onnx_path),
        input_names=["image"], output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=17,
        dynamo=False,                 # the classic exporter; onnx2torch reads its graphs
    )
print("wrote", onnx_path, f"({onnx_path.stat().st_size / 2**20:.0f} MB)")


wrote /home/space/datasets/peal/peal_runs/notebook_sim/runs_final4/bring_your_own/freight_car_resnet18.onnx (45 MB)


### Option B: your own ONNX file


In [6]:
USE_OWN_MODEL = False
if USE_OWN_MODEL:
    onnx_path = Path("/path/to/my_model.onnx")


**Check the graph the way PEAL will call it.** First with onnxruntime alone (is the batch axis
dynamic, are there two outputs?), then through `get_predictor`, PEAL's loader for every model format.
For ONNX it converts the graph into a trainable torch module with `onnx2torch` (so PEAL can also
*correct* the model later), and falls back to an inference-only onnxruntime wrapper with a warning
when an operator cannot be converted. The two must agree.


In [7]:
import onnxruntime as ort
from peal.architectures.predictors import get_predictor

options = ort.SessionOptions(); options.intra_op_num_threads = 2     # avoids thread-affinity warnings
session = ort.InferenceSession(str(onnx_path), options, providers=["CPUExecutionProvider"])
x = np.random.rand(3, 3, 224, 224).astype(np.float32)
ort_out = session.run(None, {session.get_inputs()[0].name: x})[0]
print("onnxruntime:", session.get_inputs()[0].shape, "->", ort_out.shape)
assert ort_out.shape == (3, 2), "need a dynamic batch axis and exactly two outputs"

student, _ = get_predictor(str(onnx_path), device=DEVICE)
print("PEAL loaded it as", type(student).__name__)
with torch.no_grad():
    torch_out = student(torch.from_numpy(x).to(DEVICE)).cpu().numpy()
print("max |PEAL - onnxruntime| =", float(np.abs(torch_out - ort_out).max()), "(float noise between GPU and CPU)")


onnxruntime: ['batch', 3, 224, 224] -> (3, 2)


[onnx_predictor] freight_car_resnet18.onnx converted to torch (constants moved 1, identity aliases 0, max |torch - onnxruntime| = 1.01e-06, 11684712 parameters)


PEAL loaded it as OnnxPredictor


max |PEAL - onnxruntime| = 0.002015829086303711 (float noise between GPU and CPU)


## 3. Data config: tell PEAL what the model expects

`Image2MixedDataset` reads the folder from section 1. `input_size` is the size the model sees
(images are resized to it), and `normalization` is `[mean, std]` per channel, applied after scaling
pixels to [0, 1]. `None` hands the model raw [0, 1] pixels. For the ResNet-18 that is the ImageNet
mean and std.


In [8]:
from peal.data.interfaces import DataConfig
from peal.data.dataset_factory import get_datasets

data_config = DataConfig(
    dataset_class="Image2MixedDataset",
    dataset_path=str(DATASET),
    num_samples=sum(counts.values()),
    input_type="image", output_type="singleclass",
    input_size=[3, 224, 224], output_size=[2],
    x_selection="imgs", confounding_factors=["Class"],
    split=[0.6, 0.8],                                            # 60 / 20 / 20
    normalization=[[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]],  # what YOUR model expects
)
train_set, val_set, test_set = get_datasets(data_config)
print(f"{len(train_set)} train / {len(val_set)} validation / {len(test_set)} test images")


600 train / 200 validation / 200 test images


**The final check: accuracy on your data.** A model that is right on PEAL's tensors is wired up
correctly; one at chance level usually means a swapped label order or a wrong input size. A wrong
normalization is sneakier: a robust model may stay accurate while its logits - and every explanation
built on its gradients or decisions - change. (The web demo runs the same accuracy check first on every
upload.)


In [9]:
from torch.utils.data import ConcatDataset, DataLoader

def accuracy_per_class(model, dataset):
    preds, labels = [], []
    for x, y in DataLoader(dataset, batch_size=32):
        with torch.no_grad():
            preds.append(model(x.to(DEVICE)).argmax(1).cpu())
        labels.append(y.long().flatten())
    preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
    return {"overall": (preds == labels).mean(),
            **{f"{name} (label {k})": (preds[labels == k] == k).mean()
               for k, name in enumerate([CLASS_A, CLASS_B])}}

all_images = ConcatDataset([train_set, val_set, test_set])
print(pd.Series(accuracy_per_class(student, all_images)).round(3).to_string())

# a wrong normalization does not always show in the accuracy, but it shifts what the model sees:
# compare the logits on the same images with and without the ImageNet normalization
wrong_set = get_datasets(data_config.model_copy(update={"normalization": None}))[0]
x_right, x_wrong = torch.stack([train_set[i][0] for i in range(8)]), torch.stack([wrong_set[i][0] for i in range(8)])
with torch.no_grad():
    shift = (student(x_right.to(DEVICE)) - student(x_wrong.to(DEVICE))).abs().mean().item()
print(f"\nmean logit change on the same 8 images with normalization=None: {shift:.2f}")


overall                    0.968
freight car (label 0)      0.956
passenger car (label 1)    0.980



mean logit change on the same 8 images with normalization=None: 6.20


**Save what the next notebooks need.** The data config is a pydantic model; saving it as yaml
makes it a config file like the shipped ones, loadable with `load_yaml_config`.


In [10]:
import yaml
(WORK / "data.yaml").write_text(yaml.safe_dump(data_config.model_dump(mode="json"), sort_keys=False))
print("model:", onnx_path)
print("data config:", WORK / "data.yaml")


model: /home/space/datasets/peal/peal_runs/notebook_sim/runs_final4/bring_your_own/freight_car_resnet18.onnx
data config: /home/space/datasets/peal/peal_runs/notebook_sim/runs_final4/bring_your_own/data.yaml


## Where next

* **Notebook 03** ranks the concepts this ONNX model relies on with DiDAE, without training anything.
* **Notebook 01** explains it with SCE counterfactuals from a pretrained diffusion model and can
  correct it with CFKD; the `onnx2torch` conversion above is what makes an ONNX model trainable.
* **The web demo** takes exactly these two things - the ONNX file and a zip of class folders.
